# Cross-Validation Methods Comparison Lab

Compare the empirical mechanics, runtime complexity, and variance of Leave-One-Out (LOO-CV), Leave-P-Out (LPO-CV), and standard K-Fold Cross-Validation.

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, LeaveOneOut, LeavePOut, KFold
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Leave-One-Out Cross-Validation (LOO-CV)

On a tiny dataset ($N=10$), LOO trains $N$ models where each test fold contains exactly 1 sample. Notice how each individual fold score is binary (0% or 100%), producing high fold variance.

In [ ]:
# Small dataset: 10 samples
X_small = np.random.randn(10, 2)
y_small = (X_small[:, 0] + X_small[:, 1] > 0).astype(int)

loo = LeaveOneOut()
model = LogisticRegression(random_state=42, max_iter=1000)
scores_loo = cross_val_score(model, X_small, y_small, cv=loo, scoring='accuracy')

print(f"LOO-CV on {len(X_small)} samples ({len(scores_loo)} folds):")
for i, score in enumerate(scores_loo):
    print(f"  Fold {i+1:<2} -> Test Score: {score:.1%}")

print(f"\nMean CV Score: {np.mean(scores_loo):.1%}")
print(f"Std Dev:       {np.std(scores_loo):.1%}")

## 2. Leave-P-Out Cross-Validation (LPO-CV)

LPO-CV evaluates all possible combinations $\binom{N}{P}$. Notice the combinatorial explosion: for $N=10$ and $P=2$, the number of folds jumps to 45.

In [ ]:
lpo = LeavePOut(p=2)
scores_lpo = cross_val_score(model, X_small, y_small, cv=lpo, scoring='accuracy')
n_folds_lpo = len(scores_lpo)

print(f"LPO-CV (P=2) produces C(10, 2) = {n_folds_lpo} folds")
print(f"Mean CV Score: {np.mean(scores_lpo):.1%}")
print(f"Std Dev:       {np.std(scores_lpo):.1%}")
print(f"Cost: {n_folds_lpo / len(scores_loo):.1f}x more model fits than LOO-CV for N=10!")

## 3. Practical K-Fold Cross-Validation on Iris

Evaluate how varying $K \in [2, 3, 5, 10]$ affects the train/test ratio and score variance.

In [ ]:
iris = load_iris()
X_iris = iris.data[:, :2]
y_iris = iris.target

print(f"{'K':<4} {'Folds':<6} {'Train Size':<12} {'Test Size':<12} {'Mean CV':<10} {'Std Dev':<10}")
print("-" * 58)

for K in [2, 3, 5, 10]:
    kf = KFold(n_splits=K, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_iris, y_iris, cv=kf, scoring='accuracy')
    train_sz = int(len(X_iris) * (K - 1) / K)
    test_sz = int(len(X_iris) / K)
    print(f"{K:<4} {len(scores):<6} {train_sz:<12} {test_sz:<12} {np.mean(scores):<10.1%} {np.std(scores):<10.3f}")

## 4. Runtime Benchmark: LOO-CV vs. 5-Fold CV

Measure training duration on a 100-sample dataset to observe the computational speedup of K-Fold.

In [ ]:
X_med = np.random.randn(100, 5)
y_med = (X_med[:, 0] + X_med[:, 1] > 0).astype(int)

# 5-Fold
t0 = time.time()
scores_k5 = cross_val_score(model, X_med, y_med, cv=KFold(n_splits=5))
time_k5 = time.time() - t0

# LOO-CV
t0 = time.time()
scores_loo_med = cross_val_score(model, X_med, y_med, cv=LeaveOneOut())
time_loo = time.time() - t0

print(f"5-Fold CV Time:  {time_k5:.4f}s  (Score: {np.mean(scores_k5):.1%} ± {np.std(scores_k5):.3f})")
print(f"LOO-CV Time:     {time_loo:.4f}s  (Score: {np.mean(scores_loo_med):.1%} ± {np.std(scores_loo_med):.3f})")
print(f"Speedup:         {time_loo / time_k5:.1f}x faster with 5-Fold CV!")